# Week 3 - Unsupervised Learning + Model Evaluation
## Assignments 1 & 2 + Mini Project 3: Iris Flower Clustering Project

---
### Overview & Learning Objectives
- **Focus**: Clustering, Dimensionality Reduction, Scaling
- **Topics Covered**:
  - K-Means Clustering
  - PCA (Principal Component Analysis)
  - Elbow Method for choosing K & Silhouette Analysis
  - Feature Scaling (`StandardScaler`)
  - Model saving/loading (`joblib`, `pickle`)

---
### Notebook Structure
1. **Assignment 1**: Perform K-Means on Iris dataset, find optimal K via Elbow & Silhouette methods, and visualize clusters.
2. **Assignment 2**: Apply PCA to reduce dataset dimensions (4D to 2D/3D), evaluate explained variance, and cluster in PCA space.
3. **Core Topic**: Model Saving & Loading (`joblib` and `pickle`).
4. **Mini Project 3**: *Iris Flower Clustering Project* — full pipeline, cluster-to-label comparison (Contingency Matrix, Adjusted Rand Index, NMI, Hungarian matching, and visualization).

---
## 1. Assignment 1: K-Means Clustering on Iris Dataset & Elbow Method
**Goal**: Perform K-Means clustering on the Iris flower dataset, evaluate the Elbow method across K values, and visualize clusters with centroids.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# 1. Load Iris Dataset
df_iris = pd.read_csv('iris.csv')
print(f'Iris Dataset Shape: {df_iris.shape[0]} samples, {df_iris.shape[1]} columns')
df_iris.head()

In [ ]:
# Check dataset info, summary statistics, and class counts
df_iris.info()
print('\nMissing values:\n', df_iris.isnull().sum())
print('\nSpecies Distribution:\n', df_iris['species'].value_counts())

In [ ]:
# Feature Standardization: Crucial for distance-based clustering (Euclidean distance)
features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
X = df_iris[features].values
y_true = df_iris['species'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Scaled Features Mean:', np.round(X_scaled.mean(axis=0), 4))
print('Scaled Features Std :', np.round(X_scaled.std(axis=0), 4))

In [ ]:
# Elbow Method: Calculate Inertia (WCSS) & Silhouette Scores for k = 1 to 10
k_range = range(1, 11)
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    if k >= 2:
        sil_scores.append(silhouette_score(X_scaled, km.labels_))
    else:
        sil_scores.append(None)

# Plot Elbow Curve and Silhouette Curve side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Elbow Plot (Inertia)
axes[0].plot(list(k_range), inertias, marker='o', color='#1f77b4', lw=2.2, markersize=7)
axes[0].axvline(3, color='#d62728', linestyle='--', label='Elbow at K=3')
axes[0].set_title('Elbow Method: Inertia (WCSS) vs K', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)', fontsize=11)
axes[0].set_ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=11)
axes[0].set_xticks(list(k_range))
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend()

# 2. Silhouette Score Plot
axes[1].plot(list(k_range)[1:], [s for s in sil_scores if s is not None], marker='s', color='#2ca02c', lw=2.2, markersize=7)
axes[1].axvline(3, color='#d62728', linestyle='--', label='K=3 (Score=0.459)')
axes[1].set_title('Silhouette Score vs K', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)', fontsize=11)
axes[1].set_ylabel('Silhouette Score', fontsize=11)
axes[1].set_xticks(list(k_range)[1:])
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Train K-Means with K=3
kmeans_3 = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
cluster_labels = kmeans_3.fit_predict(X_scaled)

# Centroids in original unit measurements
centroids = scaler.inverse_transform(kmeans_3.cluster_centers_)
centroids_df = pd.DataFrame(centroids, columns=features, index=['Cluster 0', 'Cluster 1', 'Cluster 2'])
print('Cluster Centroids in Original Units:')
display(centroids_df.round(2))

# Plot 2D scatter plots of original feature pairs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Petal dimensions
sns.scatterplot(
    x=df_iris['petal_length'], y=df_iris['petal_width'],
    hue=cluster_labels, palette=['#1f77b4', '#ff7f0e', '#2ca02c'],
    s=70, edgecolor='k', ax=axes[0]
)
axes[0].scatter(centroids[:, 2], centroids[:, 3], c='red', s=200, marker='X', edgecolor='k', label='Centroids')
axes[0].set_title('K-Means Clusters: Petal Dimensions', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Petal Length (cm)')
axes[0].set_ylabel('Petal Width (cm)')
axes[0].legend(title='Cluster')

# Sepal dimensions
sns.scatterplot(
    x=df_iris['sepal_length'], y=df_iris['sepal_width'],
    hue=cluster_labels, palette=['#1f77b4', '#ff7f0e', '#2ca02c'],
    s=70, edgecolor='k', ax=axes[1]
)
axes[1].scatter(centroids[:, 0], centroids[:, 1], c='red', s=200, marker='X', edgecolor='k', label='Centroids')
axes[1].set_title('K-Means Clusters: Sepal Dimensions', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sepal Length (cm)')
axes[1].set_ylabel('Sepal Width (cm)')
axes[1].legend(title='Cluster')

plt.tight_layout()
plt.show()

---
## 2. Assignment 2: Principal Component Analysis (PCA) Dimensionality Reduction
**Goal**: Apply PCA to reduce the 4-dimensional Iris dataset to 2 principal components, examine explained variance ratios, and visualize clustering in the reduced subspace.

In [ ]:
from sklearn.decomposition import PCA

# 1. Full PCA (4 components)
pca_full = PCA(n_components=4, random_state=42)
pca_full.fit(X_scaled)

var_df = pd.DataFrame({
    'Component': [f'PC{i}' for i in range(1, 5)],
    'Explained Variance Ratio': pca_full.explained_variance_ratio_,
    'Cumulative Variance': np.cumsum(pca_full.explained_variance_ratio_)
})
print('PCA Explained Variance:')
display(var_df.round(4))

# 2. Fit 2D PCA
pca_2d = PCA(n_components=2, random_state=42)
X_pca = pca_2d.fit_transform(X_scaled)
var_pc1, var_pc2 = pca_2d.explained_variance_ratio_
print(f'\nPC1 + PC2 capture {(var_pc1 + var_pc2)*100:.2f}% of total dataset variance!')

# Component loadings
loadings = pd.DataFrame(pca_2d.components_, columns=features, index=['PC1', 'PC2'])
print('\nPrincipal Component Loadings (Feature Contributions):')
display(loadings.round(4))

In [ ]:
# Visualize 2D PCA Space: Ground Truth vs K-Means Clusters
pca_centroids = pca_2d.transform(kmeans_3.cluster_centers_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ground truth species on PCA space
sns.scatterplot(
    x=X_pca[:, 0], y=X_pca[:, 1],
    hue=df_iris['species'],
    palette={'Iris-setosa': '#1f77b4', 'Iris-versicolor': '#ff7f0e', 'Iris-virginica': '#2ca02c'},
    s=70, edgecolor='k', ax=axes[0]
)
axes[0].set_title(f'Ground Truth Species in 2D PCA Space ({(var_pc1+var_pc2)*100:.1f}% Var)', fontsize=12, fontweight='bold')
axes[0].set_xlabel(f'Principal Component 1 ({var_pc1*100:.1f}%)')
axes[0].set_ylabel(f'Principal Component 2 ({var_pc2*100:.1f}%)')
axes[0].legend(title='Species')

# K-Means clusters on PCA space
sns.scatterplot(
    x=X_pca[:, 0], y=X_pca[:, 1],
    hue=cluster_labels,
    palette=['#1f77b4', '#ff7f0e', '#2ca02c'],
    s=70, edgecolor='k', ax=axes[1]
)
axes[1].scatter(pca_centroids[:, 0], pca_centroids[:, 1], c='red', s=220, marker='X', edgecolor='k', label='PCA Centroids')
axes[1].set_title('K-Means Clusters in 2D PCA Space', fontsize=12, fontweight='bold')
axes[1].set_xlabel(f'Principal Component 1 ({var_pc1*100:.1f}%)')
axes[1].set_ylabel(f'Principal Component 2 ({var_pc2*100:.1f}%)')
axes[1].legend(title='Cluster')

plt.tight_layout()
plt.show()

---
## 3. Core Topic: Model Saving and Loading (joblib vs pickle)
**Goal**: Demonstrate serializing and deserializing machine learning models and scalers using both `joblib` and `pickle`.

In [ ]:
import joblib
import pickle
import os

# Save using joblib
joblib.dump(kmeans_3, 'kmeans_iris_model.joblib')
joblib.dump(scaler, 'scaler_iris.joblib')
joblib.dump(pca_2d, 'pca_iris.joblib')
print('Successfully saved model, scaler, and PCA with joblib!')

# Save using pickle
with open('kmeans_iris.pkl', 'wb') as f:
    pickle.dump(kmeans_3, f)
print('Successfully saved model with pickle!')

# Reload with joblib
loaded_km = joblib.load('kmeans_iris_model.joblib')
loaded_sc = joblib.load('scaler_iris.joblib')

# Test inference on novel sample query
test_sample = [[5.1, 3.5, 1.4, 0.2]] # Setosa sample
pred_cluster = loaded_km.predict(loaded_sc.transform(test_sample))
print(f'Inference on test flower {test_sample[0]} -> Predicted Cluster: {pred_cluster[0]}')

# Cleanup temporary pickle
if os.path.exists('kmeans_iris.pkl'):
    os.remove('kmeans_iris.pkl')

---
## 4. Mini Project 3: Iris Flower Clustering Project
**Goal**: Complete unsupervised machine learning pipeline comparing K-Means clusters against ground truth species labels using Hungarian matching, Contingency Matrix, ARI, NMI, and multi-panel visualization.

In [ ]:
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score,
    homogeneity_score, completeness_score, v_measure_score,
    accuracy_score, classification_report, confusion_matrix
)

# 1. Contingency Matrix
contingency = pd.crosstab(df_iris['species'], cluster_labels, rownames=['True Species'], colnames=['Cluster'])
print('Contingency Matrix (True Species vs Predicted Cluster):')
display(contingency)

# 2. Optimal Bipartite Matching (Hungarian Algorithm)
classes = np.unique(y_true)
cost_matrix = np.zeros((3, len(classes)))
for c_idx in range(3):
    for l_idx, class_name in enumerate(classes):
        cost_matrix[c_idx, l_idx] = -np.sum((cluster_labels == c_idx) & (y_true == class_name))

row_ind, col_ind = linear_sum_assignment(cost_matrix)
cluster_to_label = {row: classes[col] for row, col in zip(row_ind, col_ind)}
mapped_preds = np.array([cluster_to_label[c] for c in cluster_labels])

print('Optimal Cluster-to-Species Mapping:')
for c_id, sp in cluster_to_label.items():
    print(f'  Cluster {c_id} -> {sp}')

# 3. External Validation Metrics
ari = adjusted_rand_score(y_true, cluster_labels)
nmi = normalized_mutual_info_score(y_true, cluster_labels)
homo = homogeneity_score(y_true, cluster_labels)
comp = completeness_score(y_true, cluster_labels)
v_meas = v_measure_score(y_true, cluster_labels)
acc = accuracy_score(y_true, mapped_preds)

print(f'\nAdjusted Rand Index (ARI)       : {ari:.4f}')
print(f'Normalized Mutual Info (NMI)   : {nmi:.4f}')
print(f'Homogeneity Score               : {homo:.4f}')
print(f'Completeness Score              : {comp:.4f}')
print(f'V-Measure Score                 : {v_meas:.4f}')
print(f'Mapped Clustering Accuracy      : {acc*100:.2f}%')
print('\nClassification Report:\n', classification_report(y_true, mapped_preds))

In [ ]:
# Multi-Panel Comprehensive Visualization (Saved to iris_clustering_analysis.png)
fig = plt.figure(figsize=(18, 12))
fig.suptitle('Mini Project 3: Iris Flower Clustering & PCA Evaluation', fontsize=18, fontweight='bold', y=0.98)

palette_species = {'Iris-setosa': '#1f77b4', 'Iris-versicolor': '#ff7f0e', 'Iris-virginica': '#2ca02c'}

# Subplot 1: Elbow Curve
ax1 = plt.subplot(2, 3, 1)
ax1.plot(list(k_range), inertias, marker='o', lw=2.2, color='#1f77b4', markersize=7)
ax1.axvline(3, color='#d95f02', linestyle='--', label='Optimal K=3')
ax1.set_title('1. Elbow Method (Inertia vs K)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia (WCSS)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Subplot 2: Silhouette Scores
ax2 = plt.subplot(2, 3, 2)
ax2.plot(list(k_range)[1:], [s for s in sil_scores if s is not None], marker='s', lw=2.2, color='#2ca02c', markersize=7)
ax2.axvline(3, color='#d95f02', linestyle='--', label='k=3')
ax2.set_title('2. Silhouette Analysis across K', fontsize=13, fontweight='bold')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Subplot 3: Ground Truth on Petal Dimensions
ax3 = plt.subplot(2, 3, 3)
sns.scatterplot(x=df_iris['petal_length'], y=df_iris['petal_width'], hue=df_iris['species'], palette=palette_species, s=70, edgecolor='k', ax=ax3)
ax3.set_title('3. Ground Truth: Petal Dimensions', fontsize=13, fontweight='bold')
ax3.set_xlabel('Petal Length (cm)')
ax3.set_ylabel('Petal Width (cm)')
ax3.legend(title='Species')
ax3.grid(True, alpha=0.3)

# Subplot 4: K-Means Clusters on Petal Dimensions
ax4 = plt.subplot(2, 3, 4)
sns.scatterplot(x=df_iris['petal_length'], y=df_iris['petal_width'], hue=[f'Cluster {c} ({cluster_to_label[c].replace("Iris-", "")})' for c in cluster_labels], palette=['#2b5c8f', '#d95f02', '#2ca02c'], s=70, edgecolor='k', ax=ax4)
ax4.scatter(centroids[:, 2], centroids[:, 3], c='red', s=220, marker='X', edgecolor='k', label='Centroids')
ax4.set_title('4. K-Means Clusters & Centroids (Petal Space)', fontsize=13, fontweight='bold')
ax4.set_xlabel('Petal Length (cm)')
ax4.set_ylabel('Petal Width (cm)')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Subplot 5: PCA 2D Space
ax5 = plt.subplot(2, 3, 5)
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=[f'Cluster {c}' for c in cluster_labels], palette=['#2b5c8f', '#d95f02', '#2ca02c'], s=70, edgecolor='k', ax=ax5)
ax5.scatter(pca_centroids[:, 0], pca_centroids[:, 1], c='red', s=220, marker='X', edgecolor='k', label='Centroids')
ax5.set_title(f'5. PCA 2D Projection (Clusters) - {(var_pc1+var_pc2)*100:.1f}% Var', fontsize=13, fontweight='bold')
ax5.set_xlabel(f'PC1 ({var_pc1*100:.1f}%)')
ax5.set_ylabel(f'PC2 ({var_pc2*100:.1f}%)')
ax5.legend()
ax5.grid(True, alpha=0.3)

# Subplot 6: Confusion Matrix
ax6 = plt.subplot(2, 3, 6)
cm = confusion_matrix(y_true, mapped_preds, labels=classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[s.replace('Iris-', '') for s in classes], yticklabels=[s.replace('Iris-', '') for s in classes], cbar=False, ax=ax6, annot_kws={'size': 14, 'fontweight': 'bold'})
ax6.set_title(f'6. Confusion Matrix (Accuracy: {acc*100:.1f}%)', fontsize=13, fontweight='bold')
ax6.set_xlabel('Mapped Predicted Species')
ax6.set_ylabel('True Ground Truth Species')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('iris_clustering_analysis.png', dpi=300)
plt.show()

In [ ]:
# Export Enriched Clustered Results to CSV
df_export = df_iris.copy()
df_export['cluster_id'] = cluster_labels
df_export['mapped_species'] = mapped_preds
df_export['is_correct'] = (df_export['species'] == df_export['mapped_species']).astype(int)
df_export['pca_component_1'] = np.round(X_pca[:, 0], 4)
df_export['pca_component_2'] = np.round(X_pca[:, 1], 4)
df_export.to_csv('iris_clustered_results.csv', index=False)
print(f'Successfully exported {len(df_export)} rows to iris_clustered_results.csv')
df_export.head()